# Law Topic and Title Summarization Notebook

This notebook demonstrates methods to summarize topics and titles from a dataset of laws (presumably in `nodes_df.csv`). The goal is to extract one or two key words or phrases that represent the core content of each law's topics and title, allowing for easier comparison.

**Methods Used:**
1.  **Most Frequent Phrases (for Topics):** Counts the frequency of predefined topic phrases.
2.  **TF-IDF (for Titles):** Identifies important words in titles based on their term frequency-inverse document frequency.
3.  **KeyBERT (for Topics and Titles):** Uses a small language model (Sentence Transformer) to extract keywords and keyphrases based on semantic similarity to the text.

**Steps:**
1.  Install and import necessary libraries.
2.  Load the data (sample data provided, replace with your `nodes_df.csv`).
3.  Preprocess text data (cleaning, stop-word removal).
4.  Apply summarization techniques.
5.  Display and save the results.

In [4]:
# Step 1: Install and Import Libraries
!pip install pandas scikit-learn nltk keybert sentence-transformers

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

nltk.download('punkt_tab', quiet=True)

import pandas as pd
import ast  # For safely evaluating string representations of lists
import re
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT

# Optional: For Italian lemmatization with spaCy (more advanced preprocessing)
# !python -m spacy download it_core_news_sm
# import spacy
# try:
#     nlp_it = spacy.load('it_core_news_sm')
# except OSError:
#     print("spaCy Italian model not found. Skipping lemmatization example or run the download command.")
#     nlp_it = None

print("Libraries installed and imported.")

278.50s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Libraries installed and imported.
Libraries installed and imported.


In [5]:

df = pd.read_csv('./raw_data/from_DB/nodes_df_filter.csv') # Adjust path as needed


print(df.head())

          lawId                                              title  \
0  RU 1998 2828     Ordinanza sulle banche e le casse di risparmio   
1  RU 1998 2829  Legge federale sull'assicurazione per l'invali...   
2  RU 1998 2653  Ordinanza sulle prestazioni fornite dal DPPS e...   
3  RU 1998 2656  Ordinanza concernente l'apprezzamento medico d...   
4  RU 1998 2677           Ordinanza sulla protezione civile (OPCi)   

                                              topics  
0                                                 []  
1                                                 []  
2  ['ddps', 'accordi', 'costi integrali', 'armame...  
3  ['apprezzamento medico', 'ricorsi', 'autorità ...  
4  ['protezione civile', 'servizio militare', 'ob...  


In [7]:
# Step 3: Preprocessing Functions and Application

stop_words_it = set(stopwords.words('italian'))
# Add custom stop words if they are very common and not discriminative for your task
stop_words_it.update(['legge', 'articolo', 'comma', 'federale', 'ordinanza'])

def parse_and_clean_topics(topics_str):
    """Parses a string representation of a list of topics and cleans them."""
    if not isinstance(topics_str, str):
        return []
    try:
        topics_list = ast.literal_eval(topics_str)
        if isinstance(topics_list, list):
            return [str(topic).lower().strip() for topic in topics_list if str(topic).strip()]
        return []
    except (ValueError, SyntaxError):
        return []

def preprocess_text_for_models(text, for_tfidf=False):
    """Cleans text. More aggressive cleaning for TF-IDF."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[\'"]', '', text) # Remove apostrophes and quotes
    text = re.sub(r'[^\w\s-]', ' ', text) # Keep alphanumeric, spaces, hyphens; replace other punctuation with space
    text = re.sub(r'\s+', ' ', text).strip() # Normalize whitespace
    
    if for_tfidf:
        tokens = word_tokenize(text)
        tokens = [word for word in tokens if word not in stop_words_it and word.isalpha() and len(word) > 1]
        return " ".join(tokens)
    return text

# Apply parsing and preprocessing
df['topic_phrases'] = df['topics'].apply(parse_and_clean_topics)
df['cleaned_title_keybert'] = df['title'].apply(lambda x: preprocess_text_for_models(x, for_tfidf=False))
df['processed_title_tfidf'] = df['title'].apply(lambda x: preprocess_text_for_models(x, for_tfidf=True))

# For KeyBERT on topics, join the phrases into a single string
df['joined_topic_phrases'] = df['topic_phrases'].apply(lambda x: " ".join(x))

print("\nDataFrame after preprocessing:")
print(df[['lawId', 'topic_phrases', 'joined_topic_phrases', 'cleaned_title_keybert', 'processed_title_tfidf']].head())


DataFrame after preprocessing:
          lawId                                      topic_phrases  \
0  RU 1998 2828                                                 []   
1  RU 1998 2829                                                 []   
2  RU 1998 2653  [ddps, accordi, costi integrali, armamento, fe...   
3  RU 1998 2656  [apprezzamento medico, ricorsi, autorità di ri...   
4  RU 1998 2677  [protezione civile, servizio militare, obbligh...   

                                joined_topic_phrases  \
0                                                      
1                                                      
2  ddps accordi costi integrali armamento ferrovi...   
3  apprezzamento medico ricorsi autorità di ricor...   
4  protezione civile servizio militare obblighi p...   

                               cleaned_title_keybert  \
0     ordinanza sulle banche e le casse di risparmio   
1   legge federale sullassicurazione per linvalidità   
2  ordinanza sulle prestazioni fornite dal

In [ ]:
# Step 4a: Summarize Topics - Method 1: Most Frequent Phrases

def get_top_n_phrases(phrase_list, n=2):
    if not phrase_list:
        return []
    counts = Counter(phrase_list)
    return [phrase for phrase, count in counts.most_common(n)]

df['summary_topics_freq'] = df['topic_phrases'].apply(lambda x: get_top_n_phrases(x, n=2))

print("\nSummary for Topics (Most Frequent Phrases):")
print(df[['lawId', 'topics', 'summary_topics_freq']].head())


Summary for Topics (Most Frequent Phrases):
          lawId                                             topics  \
0  RU 1998 2828                                                 []   
1  RU 1998 2829                                                 []   
2  RU 1998 2653  ['ddps', 'accordi', 'costi integrali', 'armame...   
3  RU 1998 2656  ['apprezzamento medico', 'ricorsi', 'autorità ...   
4  RU 1998 2677  ['protezione civile', 'servizio militare', 'ob...   

                         summary_topics_freq  
0                                         []  
1                                         []  
2             [telecomunicazioni, materiali]  
3  [servizio militare, idoneità al servizio]  
4     [protezione civile, servizio militare]  


In [10]:
# Step 4b: Summarize Titles - Method 1: TF-IDF

# Ensure there's content to process for TF-IDF
valid_titles_for_tfidf = df['processed_title_tfidf'][df['processed_title_tfidf'].str.strip() != ""]

if not valid_titles_for_tfidf.empty:
    tfidf_vectorizer = TfidfVectorizer(max_features=1000)
    tfidf_matrix = tfidf_vectorizer.fit_transform(valid_titles_for_tfidf)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    def get_top_tfidf_words(text_row_original_index, n=2):
        if text_row_original_index not in valid_titles_for_tfidf.index:
            return []
        matrix_idx = valid_titles_for_tfidf.index.get_loc(text_row_original_index)
        if matrix_idx >= tfidf_matrix.shape[0]:
             return []
        feature_array = tfidf_matrix[matrix_idx].toarray().flatten()
        top_indices = feature_array.argsort()[::-1]
        result_words = []
        for i in top_indices:
            if len(result_words) < n and feature_array[i] > 0 and i < len(feature_names):
                result_words.append(feature_names[i])
            elif len(result_words) >=n:
                break
        return result_words

    df['summary_title_tfidf'] = df.index.to_series().apply(lambda idx: get_top_tfidf_words(idx, n=2))
else:
    df['summary_title_tfidf'] = [[] for _ in range(len(df))]

print("\nSummary for Titles (TF-IDF):")Iprint(df[['lawId', 'title', 'processed_title_tfidf', 'summary_title_tfidf']].head())


Summary for Titles (TF-IDF):


KeyError: "['law_id'] not in index"

In [ ]:
# Step 4c: Summarize with KeyBERT (for both Topics and Titles)

kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')

def get_keybert_summary(text, n=2, ngram_range=(1,1)):
    if not text or not isinstance(text, str) or not text.strip():
        return []
    keywords = kw_model.extract_keywords(text, 
                                       keyphrase_ngram_range=ngram_range, 
                                       stop_words=list(stop_words_it),
                                       top_n=n,
                                       use_mmr=True, 
                                       diversity=0.5)
    return [kw[0] for kw in keywords]

df['summary_topics_keybert'] = df['joined_topic_phrases'].apply(lambda x: get_keybert_summary(x, n=2, ngram_range=(1,2)))
df['summary_title_keybert'] = df['cleaned_title_keybert'].apply(lambda x: get_keybert_summary(x, n=2, ngram_range=(1,1)))

print("\nSummary for Topics (KeyBERT):")
print(df[['lawId', 'joined_topic_phrases', 'summary_topics_keybert']].head())
print("\nSummary for Titles (KeyBERT):")
print(df[['lawId', 'cleaned_title_keybert', 'summary_title_keybert']].head())

In [ ]:
# Step 5: Final Output and Considerations

print("\n--- Final DataFrame with All Summaries ---")
print(df[['lawId', 'title', 'topics', 'summary_topics_freq', 'summary_topics_keybert', 'summary_title_tfidf', 'summary_title_keybert']].head(10))

print("\n--- Considerations & Next Steps ---")
print("1.  **Consistency & Quality:**")
print("    *   `summary_topics_freq`: Very consistent. Good if topic phrases are well-defined.")
print("    *   `summary_title_tfidf`: Consistent for a given dataset. Quality depends on preprocessing.")
print("    *   `summary_topics_keybert` & `summary_title_keybert`: Good semantic quality. Consistent with a fixed model.")

print("\n2.  **'One or Two Words/Phrases':** All methods are configured for `n=2`. Adjust as needed.")

print("\n3.  **Comparison:** You can now group or compare laws based on these summary columns.")

print("\n4.  **Further Improvements:** Lemmatization, custom stop words, parameter tuning for KeyBERT/TF-IDF, evaluation.")

# To save the results to a CSV file in Colab:
# df.to_csv('nodes_df_with_summaries.csv', index=False)
# try:
#     from google.colab import files
#     files.download('nodes_df_with_summaries.csv')
#     print("\nDataFrame saved to nodes_df_with_summaries.csv and download prompted.")
# except ImportError:
#     print("\nTo download the file, uncomment and run the google.colab.files part in a Colab environment.")

print("\nNotebook execution complete.")